In [5]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile

# 1. Cargar el audio original
frecuencia_muestreo, audio = wavfile.read('EpicMusic.wav')

# Convertir a flotantes normalizados entre -1 y 1
audio_norm = audio / np.max(np.abs(audio))

# PASO CLAVE: Si el audio original es Mono, lo convertimos artificialmente a Estéreo
if len(audio_norm.shape) == 1:
    print("El audio original es Mono. Duplicando canales para forzar Estéreo...")
    audio_stereo = np.vstack((audio_norm, audio_norm)).T
else:
    print("El audio original ya es Estéreo.")
    audio_stereo = audio_norm.copy()

print(f"Forma de la matriz de audio: {audio_stereo.shape} (Muestras, Canales)")
print(f"Canal 0 = Izquierdo, Canal 1 = Derecho")

El audio original ya es Estéreo.
Forma de la matriz de audio: (673792, 2) (Muestras, Canales)
Canal 0 = Izquierdo, Canal 1 = Derecho


In [23]:
# 1. Leer el audio esteganográfico recibido
fs_recibido, audio_recibido = wavfile.read('audio.wav')
audio_recibido_norm = audio_recibido / 32767.0

# 2. Extraer canal derecho y calcular FFT
canal_derecho_recibido = audio_recibido_norm[:, 1]
fhat_der_recibido = np.fft.fft(canal_derecho_recibido)
N_recibido = len(fhat_der_recibido)
magnitud_der_recibida = np.abs(fhat_der_recibido)

# 3. Localizar el índice inicial de 16 kHz
indice_inicio_recibido = int(round((16000 * N_recibido) / fs_recibido))

# 4. Leer la cantidad de bits ocultos
bits_a_recuperar = int(round(magnitud_der_recibida[indice_inicio_recibido]))
print(f"Analizando espectro...")
print(f"Cantidad de bits detectados en la FFT: {bits_a_recuperar}")

# --- PASO MATEMÁTICO 5: Decodificación por Umbral Dinámico ---
UMBRAL_DECISION = 10.0  # Punto medio entre VALOR_ALTO (20) y VALOR_BAJO (0)
bits_recuperados = ""

for i in range(bits_a_recuperar):
    idx = indice_inicio_recibido + 1 + i
    valor_espectral = magnitud_der_recibida[idx]
    
    # CORREGIDO: Ahora coincide exactamente con UMBRAL_DECISION
    if valor_espectral >= UMBRAL_DECISION:
        bits_recuperados += "1"
    else:
        bits_recuperados += "0"

# --- PASO MATEMÁTICO 6: Reconstruir el texto desde los bloques de 8 bits ---
mensaje_final = ""
for i in range(0, len(bits_recuperados), 8):
    byte = bits_recuperados[i:i+8]
    if len(byte) == 8:
        codigo_ascii = int(byte, 2)
        mensaje_final += chr(codigo_ascii)

print("\n--- ¡MENSAJE SECRETO EXTRAÍDO CON ÉXITO ABSOLUTO! ---")
print(f"Resultado: {mensaje_final}")

Analizando espectro...
Cantidad de bits detectados en la FFT: 167

--- ¡MENSAJE SECRETO EXTRAÍDO CON ÉXITO ABSOLUTO! ---
Resultado: A Dios sea la gloria
